In [49]:
import pandas as pd
import numpy as np


# ==============================
# 2. Chargement des datasets
# ==============================

# Remplace les chemins par les tiens si besoin
df_market = pd.read_csv(r"C:\Users\merry b\Downloads\market_researcher_dataset.csv")
df_data = pd.read_csv(r"C:\Users\merry b\Downloads\farmer_advisor_dataset.csv")

print("Market dataset shape :", df_market.shape)
print("Farm dataset shape :", df_data.shape)

Market dataset shape : (10000, 10)
Farm dataset shape : (10000, 10)


In [51]:
farm_dist = df_data["Crop_Type"].value_counts()

print("Distribution des cultures dans le dataset FARM :")
print(farm_dist)

market_dist = df_market["Product"].value_counts()

print("Distribution des produits dans le dataset MARKET :")
print(market_dist)

Distribution des cultures dans le dataset FARM :
Crop_Type
Soybean    2559
Wheat      2522
Rice       2464
Corn       2455
Name: count, dtype: int64
Distribution des produits dans le dataset MARKET :
Product
Rice       2561
Soybean    2514
Wheat      2475
Corn       2450
Name: count, dtype: int64


In [52]:
print("\nValeurs manquantes (Market):")
print(df_market.isnull().sum())

print("\nValeurs manquantes (Farm):")
print(df_data.isnull().sum())


Valeurs manquantes (Market):
Market_ID                   0
Product                     0
Market_Price_per_ton        0
Demand_Index                0
Supply_Index                0
Competitor_Price_per_ton    0
Economic_Indicator          0
Weather_Impact_Score        0
Seasonal_Factor             0
Consumer_Trend_Index        0
dtype: int64

Valeurs manquantes (Farm):
Farm_ID                 0
Soil_pH                 0
Soil_Moisture           0
Temperature_C           0
Rainfall_mm             0
Crop_Type               0
Fertilizer_Usage_kg     0
Pesticide_Usage_kg      0
Crop_Yield_ton          0
Sustainability_Score    0
dtype: int64


In [55]:
numeric_cols = df_market.select_dtypes(include=np.number).columns
def detect_outliers_iqr(column):
    Q1 = df_market[column].quantile(0.25)
    Q3 = df_market[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    return df_market[(df_market[column] < lower) | (df_market[column] > upper)]

for col in numeric_cols:
    outliers = detect_outliers_iqr(col)
    print(f"Outliers dans {col} :", len(outliers))



numeric_cols1 = df_data.select_dtypes(include=np.number).columns
def detect_outliers_iqr(column):
    Q1 = df_data[column].quantile(0.25)
    Q3 = df_data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    return df_data[(df_data[column] < lower) | (df_data[column] > upper)]

for col in numeric_cols1:
    outliers = detect_outliers_iqr(col)
    print(f"Outliers dans {col} :", len(outliers))

Outliers dans Market_ID : 0
Outliers dans Market_Price_per_ton : 0
Outliers dans Demand_Index : 0
Outliers dans Supply_Index : 0
Outliers dans Competitor_Price_per_ton : 0
Outliers dans Economic_Indicator : 0
Outliers dans Weather_Impact_Score : 0
Outliers dans Consumer_Trend_Index : 0
Outliers dans Farm_ID : 0
Outliers dans Soil_pH : 0
Outliers dans Soil_Moisture : 0
Outliers dans Temperature_C : 0
Outliers dans Rainfall_mm : 0
Outliers dans Fertilizer_Usage_kg : 0
Outliers dans Pesticide_Usage_kg : 0
Outliers dans Crop_Yield_ton : 0
Outliers dans Sustainability_Score : 0


In [57]:
df_market = df_market.drop_duplicates()
df_data = df_data.drop_duplicates()

In [59]:
df_market["Product"] = df_market["Product"].str.strip().str.lower()
df_data["Crop_Type"] = df_data["Crop_Type"].str.strip().str.lower()

In [61]:
import pandas as pd

# Compter les occurrences
farm_counts = df_data["Crop_Type"].value_counts()
market_counts = df_market["Product"].value_counts()

# Nombre commun par culture
common_counts = {
    crop: min(farm_counts[crop], market_counts[crop])
    for crop in farm_counts.index
}

print("Nombre retenu par culture :", common_counts)

# Équilibrage FARM
df_farm_balanced = (
    df_data
    .groupby("Crop_Type", group_keys=False)
    .apply(lambda x: x.sample(n=common_counts[x.name], random_state=42))
)

print(df_farm_balanced["Crop_Type"].value_counts())

Nombre retenu par culture : {'soybean': 2514, 'wheat': 2475, 'rice': 2464, 'corn': 2450}
Crop_Type
soybean    2514
wheat      2475
rice       2464
corn       2450
Name: count, dtype: int64


C:\Users\merry b\AppData\Local\Temp\ipykernel_25512\2315669339.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=common_counts[x.name], random_state=42))


In [62]:
df_market_balanced = (
    df_market
    .groupby("Product", group_keys=False)
    .apply(lambda x: x.sample(n=common_counts[x.name], random_state=42))
)

print(df_market_balanced["Product"].value_counts())

Product
soybean    2514
wheat      2475
rice       2464
corn       2450
Name: count, dtype: int64


C:\Users\merry b\AppData\Local\Temp\ipykernel_25512\579348906.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=common_counts[x.name], random_state=42))


In [65]:
import pandas as pd

# Ajouter un compteur cumulatif pour chaque produit
df_farm_balanced["prod_count"] = df_farm_balanced.groupby("Crop_Type").cumcount()
df_market_balanced["prod_count"] = df_market_balanced.groupby("Product").cumcount()

# Merge sur clé + compteur pour faire un 1-to-1
df_merged = pd.merge(
    df_farm_balanced,
    df_market_balanced,
    left_on=["Crop_Type", "prod_count"],
    right_on=["Product", "prod_count"],
    how="inner"
)

# Supprimer la colonne compteur si tu veux
df_merged.drop(columns=["prod_count"], inplace=True)


In [66]:
print(df_merged.shape)
print(df_merged["Product"].value_counts())
df_merged.head(10)

(9903, 20)
Product
soybean    2514
wheat      2475
rice       2464
corn       2450
Name: count, dtype: int64


,Farm_ID,Soil_pH,Soil_Moisture,Temperature_C,Rainfall_mm,Crop_Type,Fertilizer_Usage_kg,Pesticide_Usage_kg,Crop_Yield_ton,Sustainability_Score,Market_ID,Product,Market_Price_per_ton,Demand_Index,Supply_Index,Competitor_Price_per_ton,Economic_Indicator,Weather_Impact_Score,Seasonal_Factor,Consumer_Trend_Index
0,6565,7.032153,46.514269,18.913628,290.091603,corn,196.260655,8.940415,2.463432,38.120222,5312,corn,319.577395,188.755195,62.659718,468.050339,0.900259,4.929760,Low,94.345953
1,2455,5.663934,23.228243,34.527513,168.079485,corn,74.652184,2.742893,7.481589,82.231921,4094,corn,189.482608,58.908468,81.433946,230.310583,1.418724,45.069579,Medium,136.511575
2,5205,5.781744,42.657577,17.103046,157.923184,corn,94.418781,12.916802,9.466945,46.561818,9663,corn,168.560868,171.283615,151.323965,101.667492,1.324558,79.153151,Low,66.693884
3,2536,7.183501,44.621925,19.596388,248.829915,corn,128.920997,9.193298,8.249470,15.600613,2700,corn,394.530538,62.410229,68.900154,489.005138,0.645194,0.286621,Low,113.075330
4,7746,6.931830,14.209751,20.221501,154.000925,corn,143.423213,9.962443,7.554546,40.896579,2544,corn,111.373627,160.830130,87.009354,196.593936,1.070757,73.781026,Low,111.343717
5,6733,7.413270,31.108110,21.678898,130.943653,corn,149.668009,6.738812,4.716848,68.825836,6008,corn,200.397538,169.393901,134.091063,351.782906,1.215227,96.843622,Low,118.336240
6,7222,6.255509,42.576729,29.547064,76.900640,corn,153.555083,18.993588,8.646623,30.622897,6082,corn,110.522919,117.639834,50.914434,498.450107,1.019874,74.556742,Low,96.294764
7,6430,7.433210,39.250734,32.151431,269.226302,corn,191.945717,6.144577,9.744662,50.908294,8401,corn,253.764066,128.849109,79.631465,123.746602,0.824488,23.848849,Medium,139.021237
8,5343,5.680675,41.895944,26.119116,96.706908,corn,155.716730,11.185594,3.909241,83.328282,7722,corn,390.706237,92.933418,55.728794,126.195700,1.306052,10.126492,Medium,129.639782
9,3477,5.571144,19.616052,23.200443,266.405250,corn,198.319265,15.509984,1.001584,92.101932,9701,corn,196.221715,170.043593,146.619355,250.792709,1.127596,88.229186,Low,50.310711


In [68]:
cols_to_drop = [
    "Farm_ID",
    "Market_ID",
    "Crop_Type"
]
df_final = df_merged.drop(columns=cols_to_drop, errors="ignore")

In [69]:
df_final

,Soil_pH,Soil_Moisture,Temperature_C,Rainfall_mm,Fertilizer_Usage_kg,Pesticide_Usage_kg,Crop_Yield_ton,Sustainability_Score,Product,Market_Price_per_ton,Demand_Index,Supply_Index,Competitor_Price_per_ton,Economic_Indicator,Weather_Impact_Score,Seasonal_Factor,Consumer_Trend_Index
0,7.032153,46.514269,18.913628,290.091603,196.260655,8.940415,2.463432,38.120222,corn,319.577395,188.755195,62.659718,468.050339,0.900259,4.929760,Low,94.345953
1,5.663934,23.228243,34.527513,168.079485,74.652184,2.742893,7.481589,82.231921,corn,189.482608,58.908468,81.433946,230.310583,1.418724,45.069579,Medium,136.511575
2,5.781744,42.657577,17.103046,157.923184,94.418781,12.916802,9.466945,46.561818,corn,168.560868,171.283615,151.323965,101.667492,1.324558,79.153151,Low,66.693884
3,7.183501,44.621925,19.596388,248.829915,128.920997,9.193298,8.249470,15.600613,corn,394.530538,62.410229,68.900154,489.005138,0.645194,0.286621,Low,113.075330
4,6.931830,14.209751,20.221501,154.000925,143.423213,9.962443,7.554546,40.896579,corn,111.373627,160.830130,87.009354,196.593936,1.070757,73.781026,Low,111.343717
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9898,5.839200,20.896012,23.080731,188.824818,156.699151,12.070118,6.040859,97.621116,wheat,178.036094,175.002122,56.798817,233.298206,0.851559,83.947805,High,116.625816
9899,6.264563,32.110786,24.300163,286.771995,150.582637,8.443567,6.156517,24.723382,wheat,135.411514,122.097547,58.467391,343.958490,0.781635,38.901079,Medium,70.599105
9900,6.475076,25.440981,31.719444,272.051491,101.418086,10.928402,4.335814,61.590380,wheat,249.738343,117.631217,167.275192,464.676639,0.727641,43.664988,High,120.521269
9901,6.028879,12.650102,31.623344,149.834858,105.154157,17.606537,3.864660,6.459584,wheat,129.127711,198.252939,78.148846,461.692581,1.112747,0.374335,High,81.618480


In [70]:
df_final["demand_supply_ratio"] = (
    df_final["Demand_Index"] /
    (df_final["Supply_Index"] + 0.01)
)

df_final["Input_Intensity"] = (
    df_final["Fertilizer_Usage_kg"] + df_final["Pesticide_Usage_kg"]
)

df_final["Climate_Stress_Score"] = (
    df_final["Temperature_C"] / (df_final["Rainfall_mm"] + 1e-5)
)

df_final["market_stability"] = (
    df_final["Economic_Indicator"] *
    df_final["Consumer_Trend_Index"]
)

In [71]:
# ==============================
# ENCODAGE AVANCÉ
# ==============================

from sklearn.preprocessing import OrdinalEncoder
import pandas as pd


df_encoded = df_final.copy()


# ---- 1. Encodage ordinal pour la saison ----

season_order = [["Low", "Medium", "High"]]

encoder = OrdinalEncoder(categories=season_order)

df_encoded["Seasonal_Factor"] = encoder.fit_transform(
    df_encoded[["Seasonal_Factor"]]
)


# ---- 2. One-Hot pour Product ----

df_encoded = pd.get_dummies(
    df_encoded,
    columns=["Product"],
    drop_first=False
)

df_encoded

,Soil_pH,Soil_Moisture,Temperature_C,Rainfall_mm,Fertilizer_Usage_kg,Pesticide_Usage_kg,Crop_Yield_ton,Sustainability_Score,Market_Price_per_ton,Demand_Index,...,Seasonal_Factor,Consumer_Trend_Index,demand_supply_ratio,Input_Intensity,Climate_Stress_Score,market_stability,Product_corn,Product_rice,Product_soybean,Product_wheat
0,7.032153,46.514269,18.913628,290.091603,196.260655,8.940415,2.463432,38.120222,319.577395,188.755195,...,0.0,94.345953,3.011904,205.201070,0.065199,84.935811,True,False,False,False
1,5.663934,23.228243,34.527513,168.079485,74.652184,2.742893,7.481589,82.231921,189.482608,58.908468,...,1.0,136.511575,0.723301,77.395077,0.205424,193.672233,True,False,False,False
2,5.781744,42.657577,17.103046,157.923184,94.418781,12.916802,9.466945,46.561818,168.560868,171.283615,...,0.0,66.693884,1.131825,107.335583,0.108300,88.339911,True,False,False,False
3,7.183501,44.621925,19.596388,248.829915,128.920997,9.193298,8.249470,15.600613,394.530538,62.410229,...,0.0,113.075330,0.905675,138.114295,0.078754,72.955486,True,False,False,False
4,6.931830,14.209751,20.221501,154.000925,143.423213,9.962443,7.554546,40.896579,111.373627,160.830130,...,0.0,111.343717,1.848211,153.385655,0.131308,119.222077,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9898,5.839200,20.896012,23.080731,188.824818,156.699151,12.070118,6.040859,97.621116,178.036094,175.002122,...,2.0,116.625816,3.080545,168.769269,0.122234,99.313706,False,False,False,True
9899,6.264563,32.110786,24.300163,286.771995,150.582637,8.443567,6.156517,24.723382,135.411514,122.097547,...,1.0,70.599105,2.087944,159.026204,0.084737,55.182731,False,False,False,True
9900,6.475076,25.440981,31.719444,272.051491,101.418086,10.928402,4.335814,61.590380,249.738343,117.631217,...,2.0,120.521269,0.703178,112.346488,0.116594,87.696169,False,False,False,True
9901,6.028879,12.650102,31.623344,149.834858,105.154157,17.606537,3.864660,6.459584,129.127711,198.252939,...,2.0,81.618480,2.536539,122.760694,0.211055,90.820731,False,False,False,True


In [76]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

In [56]:
# Targets
y_yield = df_encoded["Crop_Yield_ton"]

# Features (ON NE MET PAS profit ni targets)
X = df_encoded.drop([
    "Crop_Yield_ton",
], axis=1)

X_train, X_test, \
y_yield_train, y_yield_test= train_test_split(

    X,
    y_yield,

    test_size=0.2,
    random_state=42
)

rf_yield = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    random_state=42
)

rf_yield.fit(X_train, y_yield_train)

y_yield_pred = rf_yield.predict(X_test)

print("=== RENDEMENT ===")
print("RMSE:", np.sqrt(mean_squared_error(y_yield_test, y_yield_pred)))
print("R2:", r2_score(y_yield_test, y_yield_pred))

=== RENDEMENT ===
RMSE: 2.626227297567766
R2: -0.005862425989790587


In [43]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Targets
y_yield = df_encoded["Crop_Yield_ton"]

# Features (ON NE MET PAS profit ni targets)
X = df_encoded.drop([
    "Crop_Yield_ton",
], axis=1)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_yield, test_size=0.2, random_state=42
)


from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# =========================
# 1. Normalisation des données (très important pour ElasticNet)
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 2. Définir le modèle ElasticNet
# =========================
# alpha = régularisation globale
# l1_ratio = proportion L1/L2 (0 = Ridge, 1 = Lasso)
model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=10000)

# =========================
# 3. Entraînement
# =========================
model.fit(X_train_scaled, y_train)

# =========================
# 4. Prédiction
# =========================
y_pred = model.predict(X_test_scaled)

# =========================
# 5. Évaluation
# =========================
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("ElasticNet Results:")
print("RMSE:", rmse)
print("R2:", r2)
print("MAE:", mae)

ElasticNet Results:
RMSE: 2.6189013170983295
R2: -0.00025845521848411046
MAE: 2.268479933398495


In [81]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

y_price = df_encoded["Market_Price_per_ton"]

# 2️⃣ Features (sans fuite)
X_price = df_encoded.drop(
    columns=[
        "Market_Price_per_ton",
    ],
    errors="ignore"
)

X_train, X_test, y_train, y_test = train_test_split(
    X_price, y_price,
    test_size=0.2,
    random_state=42
)

# -------------------------
# 7️⃣ Modèle Gradient Boosting Regressor
# -------------------------
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    random_state=42
)

# Entraînement
gbr.fit(X_train, y_train)

# Prédiction
y_pred = gbr.predict(X_test)

# -------------------------
# 8️⃣ Évaluation
# -------------------------
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))


print("=== RENTABILTE ===")
print(f"R² : {r2:.3f}")
print(f"RMSE : {rmse:.3f}")
from sklearn.metrics import mean_absolute_error
print("mae:", mean_absolute_error(y_test, y_pred))

=== RENTABILTE ===
R² : -0.017
RMSE : 117.332
mae: 101.06131883438101


In [41]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
from sklearn.metrics import mean_absolute_error

# Targets
y_price = df_encoded["Market_Price_per_ton"]

# 2️⃣ Features (sans fuite)
X_price = df_encoded.drop(
    columns=[
        "Market_Price_per_ton",
    ],
    errors="ignore"
)

X_train, X_test, y_train, y_test = train_test_split(
    X_price, y_price,
    test_size=0.2,
    random_state=42
)


from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

# =========================
# 1. Normalisation des données (très important pour ElasticNet)
# =========================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 2. Définir le modèle ElasticNet
# =========================
# alpha = régularisation globale
# l1_ratio = proportion L1/L2 (0 = Ridge, 1 = Lasso)
model = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42, max_iter=10000)

# =========================
# 3. Entraînement
# =========================
model.fit(X_train_scaled, y_train)

# =========================
# 4. Prédiction
# =========================
y_pred = model.predict(X_test_scaled)

# =========================
# 5. Évaluation
# =========================
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("ElasticNet Results:")
print("RMSE:", rmse)
print("R2:", r2)
print("MAE:", mae)

ElasticNet Results:
RMSE: 116.30365293484999
R2: 0.00029385876768250885
MAE: 100.77660833846403


In [89]:
df_encoded.to_csv("C:/Users/merry b/Downloads/df_encoded1.csv", index=False)